# Walmart E-commerce Sales Pipeline

This notebook demonstrates a small, testable ETL workflow for Walmart weekly sales. I use the supplied DataCamp inputs, keep the transformation logic in `src/pipeline.py`, and use the notebook for exploration and communication.

The original course workflow runs `SELECT * FROM grocery_sales` in a PostgreSQL-backed SQL cell. For a reproducible local run, this notebook uses the exported local sample in `data/raw/grocery_sales_sample.csv`. Replace that file with the full SQL export when running a full-scale analysis.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.pipeline import (
    extract,
    transform,
    avg_weekly_sales_per_month,
    load,
    validation,
)

SALES_PATH = ROOT / "data/raw/grocery_sales_sample.csv"
EXTRA_DATA_PATH = ROOT / "data/raw/extra_data.parquet"
CLEAN_PATH = ROOT / "data/processed/clean_data.csv"
AGG_PATH = ROOT / "data/processed/agg_data.csv"
REPORT_PATH = ROOT / "reports"
REPORT_PATH.mkdir(exist_ok=True)

## 1. Extract and merge the two sources

In [ ]:
store_data = pd.read_csv(SALES_PATH, parse_dates=["Date"])
merged_df = extract(store_data, EXTRA_DATA_PATH)
print(f"Sales rows: {len(store_data):,}")
print(f"Merged rows: {len(merged_df):,}")
merged_df.head()

## 2. Transform

The transformation fills missing numeric values with column medians, converts the sales date to a calendar month, removes rows at or below the `$10,000` threshold, and retains the seven columns needed for downstream analysis.

In [ ]:
clean_data = transform(merged_df)
print(f"Rows after the > $10,000 filter: {len(clean_data):,}")
print(f"Remaining missing values: {int(clean_data.isna().sum().sum())}")
clean_data.head()

## 3. Aggregate monthly sales

In [ ]:
agg_data = avg_weekly_sales_per_month(clean_data)
agg_data

## 4. Load and validate the outputs

In [ ]:
load(clean_data, CLEAN_PATH, agg_data, AGG_PATH)
print({
    "clean_data.csv": validation(CLEAN_PATH),
    "agg_data.csv": validation(AGG_PATH),
})

## 5. Preliminary business analysis

In [ ]:
sns.set_theme(style="whitegrid", palette="colorblind")
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=agg_data, x="Month", y="Weekly_Sales", marker="o", color="#0F766E", ax=ax)
ax.set_title("Average weekly sales by calendar month", loc="left", weight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Average weekly sales ($)")
plt.tight_layout()
plt.show()

In [ ]:
holiday_summary = (
    clean_data.groupby("IsHoliday", as_index=False)["Weekly_Sales"]
    .mean()
    .assign(Period=lambda x: x["IsHoliday"].map({0: "Regular week", 1: "Holiday week"}))
)
holiday_summary

In [ ]:
holiday_values = holiday_summary.set_index("IsHoliday")["Weekly_Sales"]
holiday_lift = (holiday_values[1] / holiday_values[0] - 1) * 100
peak_month = int(agg_data.loc[agg_data["Weekly_Sales"].idxmax(), "Month"])
print(f"Peak month in this local extract: {peak_month}")
print(f"Holiday-week average sales lift: {holiday_lift:.1f}%")
print("
These are descriptive results, not causal estimates. The holiday flag is not a controlled experiment, and the local CSV is a sample rather than the full SQL extract.")

## Interpretation and next steps

The monthly table is useful for an initial planning conversation, but it should not be used alone to set inventory levels. A stronger version would use the complete SQL extract, separate stores and departments, compare like-for-like weeks, and control for promotions, store size, CPI, and unemployment. A natural next step would be a holiday-lift model or a demand forecast with a proper time-based validation split.